# Online Retail Recommendation System

## Project Objective

The goal of this project is to analyze customer purchasing behavior and build product recommendation systems using:

1. Item-Based Collaborative Filtering (Cosine Similarity)
2. Association Rule Mining (Apriori)

The Online Retail dataset is used for transaction analysis and recommendation generation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity

from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

# Load Dataset

In [ ]:
df = pd.read_csv(
    "data.csv",
    encoding="ISO-8859-1"
)

df.head()

# Data Cleaning

The following preprocessing steps are applied:

- Remove missing CustomerID values
- Remove missing product descriptions
- Remove returned orders
- Remove invalid prices

In [ ]:
df = df.dropna(subset=["CustomerID"])
df = df.dropna(subset=["Description"])

df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]

print(df.shape)

# Exploratory Data Analysis (EDA)

## Top Selling Products

In [ ]:
top_products = (
    df.groupby("Description")["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_products

In [ ]:
plt.figure(figsize=(10,5))

top_products.sort_values().plot(kind="barh")

plt.title("Top 10 Selling Products")
plt.xlabel("Quantity Sold")

plt.show()

## Most Active Customers

In [ ]:
top_customers = (
    df.groupby("CustomerID")["InvoiceNo"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)

top_customers

In [ ]:
plt.figure(figsize=(10,5))

top_customers.sort_values().plot(kind="barh")

plt.title("Top 10 Customers by Transactions")
plt.xlabel("Number of Transactions")

plt.show()

In [ ]:
df["TotalPrice"] = (
    df["Quantity"] *
    df["UnitPrice"]
)

top_revenue_products = (
    df.groupby("Description")["TotalPrice"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_revenue_products

# Item-Based Collaborative Filtering

A customer-product interaction matrix is created and cosine similarity is used to identify similar products.

In [ ]:
customer_item_matrix = df.pivot_table(
    index="CustomerID",
    columns="Description",
    values="Quantity",
    aggfunc="sum",
    fill_value=0
)

customer_item_matrix.shape

In [ ]:
item_similarity = cosine_similarity(
    customer_item_matrix.T
)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=customer_item_matrix.columns,
    columns=customer_item_matrix.columns
)

In [ ]:
def recommend_products(product_name, n=5):

    recommendations = (
        item_similarity_df[product_name]
        .sort_values(ascending=False)
        .iloc[1:n+1]
    )

    return recommendations

## Recommendation Example

# Association Rule Mining (Apriori)

To reduce computational cost, only the Top 50 most frequently purchased products are used.

In [ ]:
top_50_products = (
    df["Description"]
    .value_counts()
    .head(50)
    .index
)

df_apriori = (
    df[df["Description"]
    .isin(top_50_products)]
)

In [ ]:
basket = (
    df_apriori
    .groupby(
        ["InvoiceNo", "Description"]
    )["Quantity"]
    .sum()
    .unstack()
    .fillna(0)
)

basket = basket.astype(bool)

basket.shape

In [ ]:
frequent_itemsets = apriori(
    basket,
    min_support=0.03,
    use_colnames=True
)

frequent_itemsets.sort_values(
    "support",
    ascending=False
).head(10)

In [ ]:
rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1
)

rules.shape

In [ ]:
rules.sort_values(
    "confidence",
    ascending=False
).head(10)

## Top Association Rules by Lift

In [ ]:
top_rules = (
    rules
    .sort_values("lift", ascending=False)
    .head(10)
)

plt.figure(figsize=(10,5))

plt.bar(
    range(len(top_rules)),
    top_rules["lift"]
)

plt.xticks(
    range(len(top_rules)),
    [
        list(x)[0]
        for x in top_rules["antecedents"]
    ],
    rotation=90
)

plt.ylabel("Lift")
plt.title(
    "Top 10 Association Rules by Lift"
)

plt.tight_layout()

plt.show()

# Conclusion

This project implemented two recommendation approaches:

1. Item-Based Collaborative Filtering using Cosine Similarity
2. Association Rule Mining using Apriori

The collaborative filtering model successfully identified similar products based on customer purchasing behavior.

The Apriori algorithm discovered strong product associations, with the strongest rule achieving a Lift score greater than 10.